In [8]:
from google.colab import drive
# Mount Google Drive/
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import zipfile
import tarfile
import shutil

# List of zip files
zip_files = [
   "/content/drive/MyDrive/annotations_v2.zip",
   "/content/drive/MyDrive/data by hand.zip",
   "/content/drive/MyDrive/train_images.zip",
   "/content/drive/MyDrive/validation_images.zip",
   "/content/drive/MyDrive/dev_gold_labels.zip",
   "/content/drive/MyDrive/dev_images.zip"



]

# Iterate through each zip file
for zip_file in zip_files:
    # Extract the filename without extension
    file_name = os.path.splitext(os.path.basename(zip_file))[0]

    # Create a directory for each file
    extract_dir = os.path.join("/content", file_name)
    os.makedirs(extract_dir, exist_ok=True)

    try:
        # Check if the file is a zip archive
        if zipfile.is_zipfile(zip_file):
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        # Check if the file is a tar archive
        elif tarfile.is_tarfile(zip_file):
            with tarfile.open(zip_file, 'r') as tar_ref:
                tar_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        else:
            print(f"Skipping {zip_file} as it is not a zip or tar archive")
    except Exception as e:
        print(f"Error extracting {zip_file}: {e}")

Extracted /content/drive/MyDrive/annotations_v2.zip to /content/annotations_v2
Extracted /content/drive/MyDrive/data by hand.zip to /content/data by hand
Extracted /content/drive/MyDrive/train_images.zip to /content/train_images
Extracted /content/drive/MyDrive/validation_images.zip to /content/validation_images
Extracted /content/drive/MyDrive/dev_gold_labels.zip to /content/dev_gold_labels
Extracted /content/drive/MyDrive/dev_images.zip to /content/dev_images


In [12]:
import json
import pandas as pd

# Load JSON file
with open('/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json', 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)



# Display as table
display(df.head())



,id,text,image,labels,link,caption
0,63226,POLL: IF YOU THINK THIS MAN IS MENTALLY ILL\nL...,prop_meme_405.png,"[Loaded Language, Smears]",https://www.facebook.com/SilentmajorityDJT/pho...,This meme is a **political attack meme** targe...
1,64328,﻿FAKE NEWS PLANDEMIC\nVOTER FRAUD\nOPEN BORDER...,prop_meme_4269.png,"[Loaded Language, Slogans, Doubt, Flag-waving,...",https://www.facebook.com/photo/?fbid=102188324...,This meme presents a classic **conspiracy theo...
2,66831,I tell you\nwhat to wear.\nwhat to eat.\nwhat ...,prop_meme_5218.png,"[Repetition, Transfer, Black-and-white Fallacy...",null,This meme features a vintage television set wi...
3,79544,"IN CONGRESS FOR 30 YEARS\n$193,400 SALARY\n\nN...",prop_meme_24585.png,[Doubt],https://www.facebook.com/ResistanceFeed/photos...,Here's a factual evaluation of the claims made...
4,70781,NOLTE: BERNIE GETS A BRIEFING ABOUT RUSSIA\nME...,prop_meme_6607.png,"[Loaded Language, Smears, Whataboutism]",null,"This meme, attributed to ""Nolte"" (likely John ..."


In [13]:
print(f"The dataset has {len(df)} records.")

The dataset has 500 records.


In [4]:
import pandas as pd
import json

# Use the DataFrame 'df' loaded in the previous cell (QhWBjnkuTheA)
# Rename the column 'gemini_caption' to 'caption'
df.rename(columns={'gemini_caption': 'caption'}, inplace=True)

# Convert DataFrame to JSON format
# Use orient='records' to get a list of dictionaries, which is a common JSON format for tabular data
json_data = df.to_dict(orient='records')

# Define the output path for the JSON file
output_path = '/content/drive/MyDrive/validation_caption.json'

# Save the JSON data to a file
with open(output_path, 'w') as f:
    json.dump(json_data, f, indent=4) # Use indent for pretty printing

print(f"Updated DataFrame saved to: {output_path}")

Updated DataFrame saved to: /content/drive/MyDrive/validation_caption.json


In [ ]:
 {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

In [ ]:
    train_json = '/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json'
    val_json = '/content/drive/MyDrive/validation_caption.json'
    test_json = '/content/drive/MyDrive/dev_processed.json'

    train_img_dir = '/content/train_images/train_images'
    val_img_dir   = '/content/validation_images/validation_images'
    test_img_dir  = '/content/dev_images/dev_images'

In [11]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
import spacy
import re

from transformers import CLIPModel, CLIPProcessor, RobertaModel, RobertaTokenizer
import warnings
warnings.filterwarnings('ignore')

# Configuration
class CFG:
    train_json = '/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json'
    val_json = '/content/drive/MyDrive/validation_caption.json'
    test_json = '/content/drive/MyDrive/dev_processed.json'
    train_img_dir = '/content/train_images/train_images'
    val_img_dir = '/content/validation_images/validation_images'
    test_img_dir = '/content/dev_images/dev_images'

    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 32
    lr = 2e-3
    epochs = 12
    validate_every = 100
    use_caption = True
    caption_separator = " [SEP] "
    checkpoint_dir = './checkpoints'
    log_dir = './logs'
    clip_model_name = "openai/clip-vit-base-patch32"
    roberta_model_name = "roberta-base"
    gradient_clip_norm = 1.0
    early_stopping_patience = 5
    lr_scheduler_patience = 2
    lr_scheduler_factor = 0.5
    use_gcn = True
    gcn_hidden_dim = 256
    gcn_layers = 2
    pmi_weight = 0.6
    semantic_weight = 0.4
    use_kg_refinement = True

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

HIERARCHY_GRAPH = {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

TECHNIQUE_DEFINITIONS = {
    "Name calling/Labeling": "Giving a person or idea a bad label to make the audience reject them without examining evidence",
    "Repetition": "Repeating the same message over and over again so that the audience will accept it",
    "Slogans": "A brief and striking phrase that contains labeling and stereotyping",
    "Appeal to fear/prejudice": "Seeking to build support by instilling anxiety and panic in the population",
    "Doubt": "Questioning the credibility of someone or something",
    "Exaggeration/Minimisation": "Either representing something in an excessive manner or making something seem less important",
    "Flag-waving": "Playing on strong national feeling to justify or promote an action",
    "Causal Oversimplification": "Assuming a single cause when there are multiple causes behind an issue",
    "Appeal to authority": "Supposing that a claim is true because a valid authority or expert on the issue said it",
    "Black-and-white Fallacy/Dictatorship": "Presenting two alternative options as the only possibilities",
    "Thought-terminating cliché": "Words or phrases that discourage critical thought and useful discussion",
    "Whataboutism": "Discredit an opponent's position by charging them with hypocrisy without refuting their argument",
    "Reductio ad hitlerum": "Comparing something/someone to Hitler or Nazism to make the argument seem invalid",
    "Bandwagon": "Attempting to persuade the target audience to join in and take the course of action because everyone else is doing so",
    "Obfuscation, Intentional vagueness, Confusion": "Using deliberately unclear words to make the message confusing",
    "Loaded Language": "Using specific words and phrases with strong emotional implications to influence the audience",
    "Glittering generalities (Virtue)": "Words associated with highly valued concepts that are used to evoke positive emotional response",
    "Misrepresentation of Someone's Position (Straw Man)": "When an opponent's proposition is substituted with a similar one",
    "Presenting Irrelevant Data (Red Herring)": "Introducing irrelevant material to the argument to distract",
    "Transfer": "Projecting positive or negative qualities of a person, entity, object to another",
    "Appeal to (Strong) Emotions": "Attempting to develop an emotional response instead of a valid or compelling argument",
    "Smears": "A direct attack on the reputation or character of a person or group",
}

def create_label_mappings():
    all_labels = set(HIERARCHY_GRAPH.keys())
    for children in HIERARCHY_GRAPH.values():
        all_labels.update(children)

    label_to_idx = {label: i for i, label in enumerate(sorted(all_labels))}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    num_labels = len(label_to_idx)

    ancestors = {label_to_idx['Persuasion']: {label_to_idx['Persuasion']}}

    for parent, children in HIERARCHY_GRAPH.items():
        parent_idx = label_to_idx[parent]
        for child in children:
            child_idx = label_to_idx[child]
            ancestors[child_idx] = ancestors.get(child_idx, set()) | ancestors.get(parent_idx, set()) | {child_idx}

    ancestor_matrix = torch.zeros((num_labels, num_labels), dtype=torch.float32)
    for node, anc_set in ancestors.items():
        ancestor_matrix[node, list(anc_set)] = 1.0

    return label_to_idx, idx_to_label, ancestor_matrix, num_labels

def compute_pmi_matrix(labels_data, label_to_idx, smooth=1e-5):
    num_labels = len(label_to_idx)
    co_occurrence = np.zeros((num_labels, num_labels))
    label_counts = np.zeros(num_labels)
    total_samples = len(labels_data)

    for labels in tqdm(labels_data, desc="Computing co-occurrences"):
        label_indices = [label_to_idx[label] for label in labels if label in label_to_idx]
        for idx in label_indices:
            label_counts[idx] += 1
        for i in label_indices:
            for j in label_indices:
                co_occurrence[i, j] += 1

    pmi_matrix = np.zeros((num_labels, num_labels))
    for i in range(num_labels):
        for j in range(num_labels):
            p_i = (label_counts[i] + smooth) / total_samples
            p_j = (label_counts[j] + smooth) / total_samples
            p_ij = (co_occurrence[i, j] + smooth) / total_samples
            pmi = np.log(p_ij / (p_i * p_j))
            pmi_matrix[i, j] = max(0, pmi)

    if pmi_matrix.max() > 0:
        pmi_matrix = pmi_matrix / pmi_matrix.max()

    return pmi_matrix

def compute_semantic_similarity(label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device):
    num_labels = len(label_to_idx)
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(num_labels), desc="Extracting embeddings"):
            label_name = idx_to_label[i]
            definition = TECHNIQUE_DEFINITIONS.get(label_name, label_name)
            encoded = roberta_tokenizer(definition, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            outputs = roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(embedding)

    embeddings = np.array(embeddings)
    similarity_matrix = cosine_similarity(embeddings)
    similarity_matrix = (similarity_matrix + 1) / 2
    return similarity_matrix

def build_label_graph(train_df, label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device, pmi_weight=0.6, semantic_weight=0.4):
    labels_data = train_df['labels'].tolist()
    pmi_matrix = compute_pmi_matrix(labels_data, label_to_idx)
    semantic_matrix = compute_semantic_similarity(label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device)
    adjacency_matrix = pmi_weight * pmi_matrix + semantic_weight * semantic_matrix
    np.fill_diagonal(adjacency_matrix, 1.0)
    degree = adjacency_matrix.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.0
    degree_matrix_inv_sqrt = np.diag(degree_inv_sqrt)
    adjacency_matrix_norm = degree_matrix_inv_sqrt @ adjacency_matrix @ degree_matrix_inv_sqrt
    return torch.FloatTensor(adjacency_matrix_norm)

class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.counter = 0
        self.best_weights = None
        self.early_stop = False

    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.restore_best_weights:
                    model.load_state_dict(self.best_weights)
        else:
            self.best_score = score
            self.counter = 0
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

class GraphConvolutionLayer(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolutionLayer, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, input_features, adjacency_matrix):
        support = torch.matmul(input_features, self.weight)
        output = torch.einsum('ij,bjf->bif', adjacency_matrix, support)
        if self.bias is not None:
            output = output + self.bias
        return output

class LabelGCN(nn.Module):
    def __init__(self, num_labels, hidden_dim=256, num_layers=2, dropout=0.3):
        super(LabelGCN, self).__init__()
        self.num_labels = num_labels
        self.num_layers = num_layers
        self.gcn_layers = nn.ModuleList()
        self.gcn_layers.append(GraphConvolutionLayer(1, hidden_dim))
        for _ in range(num_layers - 2):
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, hidden_dim))
        if num_layers > 1:
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, 1))
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers - 1)])
        self.residual_weight = nn.Parameter(torch.FloatTensor([0.5]))

    def forward(self, predictions, adjacency_matrix):
        batch_size = predictions.size(0)
        x = predictions.unsqueeze(-1)
        for i, gcn_layer in enumerate(self.gcn_layers):
            x = gcn_layer(x, adjacency_matrix)
            if i < len(self.gcn_layers) - 1:
                x = x.transpose(1, 2)
                x = self.batch_norm[i](x)
                x = x.transpose(1, 2)
                x = self.relu(x)
                x = self.dropout(x)
        refined = x.squeeze(-1)
        alpha = torch.sigmoid(self.residual_weight)
        output = alpha * predictions + (1 - alpha) * refined
        return output

class MemeDataset(Dataset):
    def __init__(self, df, img_dir, processor, label_to_idx, ancestor_matrix, is_test=False, use_caption=True, caption_separator=" [SEP] "):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.label_to_idx = label_to_idx
        self.ancestor_matrix = ancestor_matrix
        self.num_labels = len(label_to_idx)
        self.is_test = is_test
        self.use_caption = use_caption
        self.caption_separator = caption_separator

    def __len__(self):
        return len(self.df)

    def _encode_labels(self, label_list):
        if not label_list:
            return torch.zeros(self.num_labels)
        label_indices = [self.label_to_idx[label] for label in label_list if label in self.label_to_idx]
        expanded_indices = set()
        for idx in label_indices:
            ancestors = torch.where(self.ancestor_matrix[idx] == 1)[0].tolist()
            expanded_indices.update(ancestors)
        y = torch.zeros(self.num_labels)
        y[list(expanded_indices)] = 1.0
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['text'] if 'text' in row and pd.notna(row['text']) else ""
        if self.use_caption and 'caption' in row and pd.notna(row['caption']):
            caption = row['caption']
            combined_text = f"{text}{self.caption_separator}{caption}"
        else:
            combined_text = text
        image = None
        if 'image' in row:
            img_path = self.img_dir / row['image']
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception as e:
                if not str(img_path).startswith('path/to/'):
                    print(f"Error loading image {img_path}: {e}")
                image = None
        if image is None:
            colors = ['white', 'lightgray', 'lightblue', 'lightgreen', 'lightyellow', 'lightpink']
            color = random.choice(colors)
            image = Image.new('RGB', (224, 224), color=color)
        if self.is_test or 'labels' not in row:
            labels = torch.zeros(self.num_labels)
        else:
            labels = self._encode_labels(row['labels'])
        return combined_text, image, labels, idx

class ImprovedMultiHeadMLP(nn.Module):
    def __init__(self, input_dim=1792, num_labels=22, use_gcn=True, gcn_hidden_dim=256, gcn_layers=2):
        super(ImprovedMultiHeadMLP, self).__init__()
        self.use_gcn = use_gcn
        self.layer1 = nn.Linear(input_dim, 768)
        self.layer1_bn = nn.BatchNorm1d(768)
        self.layer2 = nn.Linear(768, 512)
        self.layer2_bn = nn.BatchNorm1d(512)
        self.head1 = nn.Linear(512, 3)
        self.head1_reducer = nn.Linear(512, 64)
        self.head1_to_head2_residual = nn.Linear(512, 128)
        self.layer3 = nn.Linear(512, 256)
        self.layer3_bn = nn.BatchNorm1d(256)
        self.layer4 = nn.Linear(256, 128)
        self.layer4_bn = nn.BatchNorm1d(128)
        self.head2 = nn.Linear(128, 5)
        self.head2_reducer = nn.Linear(128, 64)
        self.head2_to_final_residual = nn.Linear(128, 64)
        self.final_layer1 = nn.Linear(128, 96)
        self.final_layer1_bn = nn.BatchNorm1d(96)
        self.final_layer2 = nn.Linear(96, 64)
        self.final_layer2_bn = nn.BatchNorm1d(64)
        self.final_residual = nn.Linear(128, 64)
        self.final_head = nn.Linear(64, num_labels)
        if self.use_gcn:
            self.gcn = LabelGCN(num_labels=num_labels, hidden_dim=gcn_hidden_dim, num_layers=gcn_layers, dropout=0.3)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adjacency_matrix=None, training_phase='all'):
        x1 = self.relu(self.layer1_bn(self.layer1(x)))
        x1 = self.dropout(x1)
        x1 = self.relu(self.layer2_bn(self.layer2(x1)))
        x1 = self.dropout(x1)
        output1 = self.head1(x1)
        head1_features = self.relu(self.head1_reducer(x1))
        head1_features = self.dropout(head1_features)
        if training_phase in ['head2', 'head3', 'all']:
            if training_phase == 'head2':
                x2_input = x1.detach()
            else:
                x2_input = x1
            x2 = self.relu(self.layer3_bn(self.layer3(x2_input)))
            x2 = self.dropout(x2)
            x2 = self.relu(self.layer4_bn(self.layer4(x2)))
            x2 = self.dropout(x2)
            head1_residual = self.head1_to_head2_residual(x1)
            if training_phase == 'head2':
                head1_residual = head1_residual.detach()
            x2 = x2 + head1_residual
            output2 = self.head2(x2)
            head2_features = self.relu(self.head2_reducer(x2))
            head2_features = self.dropout(head2_features)
        else:
            output2 = torch.zeros(x.size(0), 5, device=x.device)
            head2_features = torch.zeros(x.size(0), 64, device=x.device)
            x2 = torch.zeros(x.size(0), 128, device=x.device)
        if training_phase in ['head3', 'all']:
            if training_phase == 'head3':
                combined_features = torch.cat([head1_features.detach(), head2_features.detach()], dim=1)
                final_residual_input = torch.cat([head1_features.detach(), head2_features.detach()], dim=1)
            else:
                combined_features = torch.cat([head1_features, head2_features], dim=1)
                final_residual_input = combined_features
            x3 = self.relu(self.final_layer1_bn(self.final_layer1(combined_features)))
            x3 = self.dropout(x3)
            x3 = self.relu(self.final_layer2_bn(self.final_layer2(x3)))
            x3 = self.dropout(x3)
            final_residual = self.final_residual(final_residual_input)
            x3 = x3 + final_residual
            output_final = self.final_head(x3)
            if self.use_gcn and adjacency_matrix is not None:
                output_final_refined = self.gcn(output_final, adjacency_matrix)
                return output1, output2, output_final_refined
            else:
                return output1, output2, output_final
        else:
            output_final = torch.zeros(x.size(0), self.final_head.out_features, device=x.device)
            return output1, output2, output_final

    def freeze_head1_layers(self):
        for param in [self.layer1.parameters(), self.layer1_bn.parameters(), self.layer2.parameters(), self.layer2_bn.parameters(), self.head1.parameters(), self.head1_reducer.parameters(), self.head1_to_head2_residual.parameters()]:
            for p in param:
                p.requires_grad = False

    def unfreeze_head1_layers(self):
        for param in [self.layer1.parameters(), self.layer1_bn.parameters(), self.layer2.parameters(), self.layer2_bn.parameters(), self.head1.parameters(), self.head1_reducer.parameters(), self.head1_to_head2_residual.parameters()]:
            for p in param:
                p.requires_grad = True

    def freeze_head2_layers(self):
        for param in [self.layer3.parameters(), self.layer3_bn.parameters(), self.layer4.parameters(), self.layer4_bn.parameters(), self.head2.parameters(), self.head2_reducer.parameters(), self.head2_to_final_residual.parameters()]:
            for p in param:
                p.requires_grad = False

    def unfreeze_head2_layers(self):
        for param in [self.layer3.parameters(), self.layer3_bn.parameters(), self.layer4.parameters(), self.layer4_bn.parameters(), self.head2.parameters(), self.head2_reducer.parameters(), self.head2_to_final_residual.parameters()]:
            for p in param:
                p.requires_grad = True

    def freeze_head3_layers(self):
        for param in [self.final_layer1.parameters(), self.final_layer1_bn.parameters(), self.final_layer2.parameters(), self.final_layer2_bn.parameters(), self.final_head.parameters(), self.final_residual.parameters()]:
            for p in param:
                p.requires_grad = False
        if self.use_gcn:
            for param in self.gcn.parameters():
                param.requires_grad = False

    def unfreeze_head3_layers(self):
        for param in [self.final_layer1.parameters(), self.final_layer1_bn.parameters(), self.final_layer2.parameters(), self.final_layer2_bn.parameters(), self.final_head.parameters(), self.final_residual.parameters()]:
            for p in param:
                p.requires_grad = True
        if self.use_gcn:
            for param in self.gcn.parameters():
                param.requires_grad = True

def hierarchical_f1_score(y_pred_logits, y_true, ancestor_matrix, threshold=0.0, beta=1.0):
    y_pred = (y_pred_logits > threshold).float()
    y_true_expanded = torch.clamp(y_true @ ancestor_matrix, 0, 1)
    y_pred_expanded = torch.clamp(y_pred @ ancestor_matrix, 0, 1)
    tp = (y_true_expanded * y_pred_expanded).sum()
    true_sum = y_true_expanded.sum()
    pred_sum = y_pred_expanded.sum()
    if true_sum == 0 and pred_sum == 0:
        return 1.0, 1.0, 1.0
    elif true_sum == 0:
        return 0.0, 0.0, 0.0
    elif pred_sum == 0:
        return 0.0, 0.0, 0.0
    hierarchical_recall = tp / true_sum
    hierarchical_precision = tp / pred_sum
    if hierarchical_recall + hierarchical_precision == 0:
        hierarchical_f1 = 0.0
    else:
        hierarchical_f1 = (1 + beta**2) * hierarchical_recall * hierarchical_precision / (hierarchical_recall + beta**2 * hierarchical_precision)
    return hierarchical_f1.item(), hierarchical_recall.item(), hierarchical_precision.item()

def hierarchical_consistency_loss(predictions, ancestor_matrix, lambda_consistency=0.1):
    probs = torch.sigmoid(predictions)
    batch_size, num_labels = probs.shape
    consistency_violations = 0
    total_pairs = 0
    for i in range(num_labels):
        ancestors = torch.where(ancestor_matrix[i] == 1)[0]
        for ancestor in ancestors:
            if ancestor != i:
                violation = torch.relu(probs[:, i] - probs[:, ancestor])
                consistency_violations += violation.sum()
                total_pairs += batch_size
    if total_pairs > 0:
        avg_violation = consistency_violations / total_pairs
        return lambda_consistency * avg_violation
    else:
        return torch.tensor(0.0, device=predictions.device, requires_grad=True)

def find_optimal_thresholds(y_true, y_pred_probs, num_classes):
    optimal_thresholds = np.zeros(num_classes)
    for i in range(num_classes):
        if y_true[:, i].sum() > 0:
            precision, recall, thresholds = precision_recall_curve(y_true[:, i], y_pred_probs[:, i])
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
            best_threshold_idx = np.argmax(f1_scores)
            if best_threshold_idx < len(thresholds):
                optimal_thresholds[i] = thresholds[best_threshold_idx]
            else:
                optimal_thresholds[i] = 0.5
        else:
            optimal_thresholds[i] = 0.5
    return optimal_thresholds

class PropagandaKnowledgeGraph:
    def __init__(self, spacy_model="en_core_web_sm"):
        try:
            self.nlp = spacy.load(spacy_model)
        except:
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", spacy_model])
            self.nlp = spacy.load(spacy_model)

        self.all_techniques = [
            'Appeal to (Strong) Emotions', 'Appeal to authority', 'Appeal to fear/prejudice',
            'Bandwagon', 'Black-and-white Fallacy/Dictatorship', 'Causal Oversimplification',
            'Doubt', 'Exaggeration/Minimisation', 'Flag-waving', 'Glittering generalities (Virtue)',
            'Loaded Language', "Misrepresentation of Someone's Position (Straw Man)", 'Name calling/Labeling',
            'Obfuscation, Intentional vagueness, Confusion', 'Presenting Irrelevant Data (Red Herring)',
            'Reductio ad hitlerum', 'Repetition', 'Slogans', 'Smears', 'Thought-terminating cliché',
            'Transfer', 'Whataboutism'
        ]
        self._define_prerequisites()
        self._define_mutual_exclusions()
        self._define_co_occurrences()

    def _define_prerequisites(self):
        self.prerequisites = {
            'Appeal to (Strong) Emotions': {
                'emotion_words': ['love', 'hate', 'anger', 'joy', 'fear', 'disgust', 'sad', 'happy', 'furious', 'delighted', 'terrified', 'heartbroken'],
                'intensity_words': ['very', 'extremely', 'absolutely', 'totally', 'completely'],
                'min_emotion_intensity': 2,
            },
            'Appeal to authority': {
                'authority_titles': ['dr', 'doctor', 'professor', 'prof', 'expert', 'scientist', 'researcher', 'specialist', 'scholar', 'phd', 'md'],
                'authority_verbs': ['says', 'claims', 'states', 'argues', 'confirms', 'believes', 'recommends', 'suggests', 'warns', 'announces'],
                'needs_person': True,
                'needs_statement': True,
            },
            'Appeal to fear/prejudice': {
                'fear_words': ['danger', 'threat', 'fear', 'terror', 'scary', 'risk', 'destroy', 'collapse', 'disaster', 'catastrophe', 'crisis'],
                'conditional_words': ['if', 'unless', 'without', 'will', 'would'],
                'negative_outcome': True,
            },
            'Bandwagon': {
                'crowd_words': ['everyone', 'everybody', 'all', 'most people', 'majority', 'popular', 'trend', 'join', 'together', 'us'],
                'action_verbs': ['join', 'follow', 'support', 'believe', 'do', 'choose'],
                'needs_collective': True,
            },
            'Black-and-white Fallacy/Dictatorship': {
                'binary_patterns': [r'\beither\s+\w+\s+or\b', r'\bonly two\b', r'\bmust choose\b'],
                'dichotomy_words': ['or', 'versus', 'vs', 'against', 'opposite'],
                'exclusion_phrases': ['no other', 'no alternative', 'no middle ground'],
            },
            'Causal Oversimplification': {
                'causal_words': ['because', 'cause', 'causes', 'leads to', 'results in', 'due to', 'reason', 'blame'],
                'limiting_words': ['only', 'just', 'simply', 'solely', 'merely', 'alone'],
                'needs_causality': True,
                'single_cause_required': True,
            },
            'Doubt': {
                'doubt_words': ['doubt', 'question', 'suspect', 'uncertain', 'unclear', 'allegedly', 'supposedly', 'claims', 'so-called'],
                'questioning_patterns': [r'\?', r'really\?', r'truly\?'],
                'skeptical_phrases': ['hard to believe', 'seems unlikely', 'questionable'],
            },
            'Exaggeration/Minimisation': {
                'exaggeration_words': ['always', 'never', 'all', 'none', 'everyone', 'nobody', 'completely', 'totally', 'absolutely', 'enormous', 'tiny'],
                'superlatives': ['best', 'worst', 'greatest', 'least', 'most', 'biggest', 'smallest'],
                'intensifiers': ['very', 'extremely', 'incredibly', 'utterly', 'hardly'],
            },
            'Flag-waving': {
                'patriotic_words': ['country', 'nation', 'patriot', 'flag', 'national', 'homeland', 'motherland', 'fatherland', 'america', 'american'],
                'collective_identity': ['we', 'us', 'our', 'together'],
                'needs_visual_flag': True,
            },
            'Glittering generalities (Virtue)': {
                'virtue_words': ['freedom', 'liberty', 'democracy', 'justice', 'truth', 'honor', 'dignity', 'peace', 'progress', 'hope', 'values'],
                'vague_positive': True,
                'lacks_specifics': True,
            },
            'Loaded Language': {
                'loaded_positive': ['hero', 'brave', 'champion', 'defender', 'savior'],
                'loaded_negative': ['criminal', 'terrorist', 'corrupt', 'evil', 'traitor'],
                'emotional_charge': True,
                'min_loaded_words': 2,
            },
            "Misrepresentation of Someone's Position (Straw Man)": {
                'opponent_reference': ['they say', 'they claim', 'their argument', 'their position'],
                'distortion_indicators': ['really means', 'actually wants', 'trying to'],
                'needs_opponent': True,
                'needs_reframing': True,
            },
            'Name calling/Labeling': {
                'negative_labels': ['fool', 'idiot', 'liar', 'crook', 'corrupt', 'fake', 'traitor', 'extremist', 'radical', 'puppet'],
                'direct_attack': True,
                'personal_rather_than_policy': True,
            },
            'Obfuscation, Intentional vagueness, Confusion': {
                'vague_words': ['some', 'many', 'various', 'certain', 'particular', 'numerous'],
                'unclear_referents': ['this', 'that', 'it', 'they'],
                'complex_jargon': True,
                'lacks_clarity': True,
            },
            'Presenting Irrelevant Data (Red Herring)': {
                'topic_shift_indicators': ['but', 'however', 'meanwhile', 'by the way', 'speaking of'],
                'needs_two_topics': True,
                'irrelevance_markers': True,
            },
            'Reductio ad hitlerum': {
                'nazi_references': ['hitler', 'nazi', 'nazism', 'fascist', 'gestapo', 'holocaust'],
                'comparison_words': ['like', 'similar', 'reminds', 'just as'],
                'needs_comparison': True,
            },
            'Repetition': {
                'needs_repeated_phrase': True,
                'min_repetitions': 2,
                'similar_sentences': True,
            },
            'Slogans': {
                'short_phrase': True,
                'max_words': 10,
                'memorable_pattern': True,
                'slogan_indicators': ['!', 'remember', 'vote', 'make'],
            },
            'Smears': {
                'attack_words': ['corrupt', 'dishonest', 'fraud', 'scandal', 'crime', 'illegal', 'unethical', 'immoral'],
                'reputation_attack': True,
                'character_focus': True,
            },
            'Thought-terminating cliché': {
                'cliche_phrases': ['it is what it is', 'boys will be boys', 'at the end of the day', 'everything happens for a reason', "it's tradition"],
                'stops_discussion': True,
                'simplistic_dismissal': True,
            },
            'Transfer': {
                'needs_visual_symbol': True,
                'symbol_types': ['flag', 'logo', 'religious', 'icon', 'celebrity'],
                'association_words': ['with', 'like', 'represents', 'stands for'],
            },
            'Whataboutism': {
                'whatabout_pattern': r'\bwhat about\b|\bhow about\b|\bbut what\b',
                'deflection_words': ['but', 'however', 'instead', 'rather'],
                'needs_counter_example': True,
                'avoids_original_issue': True,
            }
        }

    def _define_mutual_exclusions(self):
        self.mutual_exclusions = {
            'Causal Oversimplification': ['Black-and-white Fallacy/Dictatorship'],
            'Black-and-white Fallacy/Dictatorship': ['Causal Oversimplification'],
            'Appeal to authority': ['Doubt', 'Reductio ad hitlerum'],
            'Doubt': ['Appeal to authority'],
            'Glittering generalities (Virtue)': ['Name calling/Labeling', 'Smears'],
            'Name calling/Labeling': ['Glittering generalities (Virtue)', 'Appeal to authority'],
            'Smears': ['Glittering generalities (Virtue)', 'Appeal to authority'],
            'Obfuscation, Intentional vagueness, Confusion': ['Slogans', 'Repetition'],
            'Slogans': ['Obfuscation, Intentional vagueness, Confusion'],
            'Appeal to authority': ['Obfuscation, Intentional vagueness, Confusion'],
            'Appeal to (Strong) Emotions': ['Causal Oversimplification'],
            'Reductio ad hitlerum': ['Appeal to authority', 'Glittering generalities (Virtue)']
        }

    def _define_co_occurrences(self):
        self.co_occurrences = {
            'Name calling/Labeling': {'often_with': ['Loaded Language', 'Smears', 'Doubt'], 'boost_factor': 1.5},
            'Loaded Language': {'often_with': ['Name calling/Labeling', 'Smears', 'Appeal to (Strong) Emotions'], 'boost_factor': 1.4},
            'Smears': {'often_with': ['Name calling/Labeling', 'Loaded Language', 'Doubt'], 'boost_factor': 1.5},
            'Appeal to fear/prejudice': {'often_with': ['Exaggeration/Minimisation', 'Appeal to (Strong) Emotions'], 'boost_factor': 1.6},
            'Exaggeration/Minimisation': {'often_with': ['Appeal to fear/prejudice', 'Loaded Language'], 'boost_factor': 1.4},
            'Flag-waving': {'often_with': ['Transfer', 'Glittering generalities (Virtue)', 'Bandwagon'], 'boost_factor': 1.7},
            'Transfer': {'often_with': ['Flag-waving', 'Appeal to (Strong) Emotions'], 'boost_factor': 1.6},
            'Black-and-white Fallacy/Dictatorship': {'often_with': ['Thought-terminating cliché'], 'boost_factor': 1.3},
            'Whataboutism': {'often_with': ['Presenting Irrelevant Data (Red Herring)', "Misrepresentation of Someone's Position (Straw Man)"], 'boost_factor': 1.5},
            'Presenting Irrelevant Data (Red Herring)': {'often_with': ['Whataboutism', 'Obfuscation, Intentional vagueness, Confusion'], 'boost_factor': 1.4},
            'Glittering generalities (Virtue)': {'often_with': ['Flag-waving', 'Bandwagon', 'Slogans'], 'boost_factor': 1.5},
            'Bandwagon': {'often_with': ['Glittering generalities (Virtue)', 'Flag-waving', 'Repetition'], 'boost_factor': 1.4},
            'Repetition': {'often_with': ['Slogans', 'Bandwagon'], 'boost_factor': 1.3},
            'Slogans': {'often_with': ['Repetition', 'Loaded Language', 'Glittering generalities (Virtue)'], 'boost_factor': 1.4}
        }

    def check_prerequisites(self, technique, text, has_person=False, has_visual_symbol=False, has_flag=False):
        if technique not in self.prerequisites:
            return {'satisfied': True, 'confidence': 1.0, 'reason': 'No prerequisites defined', 'evidence': []}

        prereq = self.prerequisites[technique]
        doc = self.nlp(text)
        text_lower = text.lower()
        confidence = 0.0
        evidence = []
        satisfied = False

        if technique == 'Appeal to (Strong) Emotions':
            emotion_count = sum(1 for word in prereq['emotion_words'] if word in text_lower)
            intensity_count = sum(1 for word in prereq['intensity_words'] if word in text_lower)
            if emotion_count >= prereq['min_emotion_intensity']:
                confidence += 0.6
                evidence.append(f"{emotion_count} emotion words found")
            if intensity_count > 0:
                confidence += 0.4
                evidence.append(f"{intensity_count} intensity words found")
            satisfied = confidence >= 0.5

        elif technique == 'Appeal to authority':
            has_title = any(title in text_lower for title in prereq['authority_titles'])
            has_verb = any(verb in text_lower for verb in prereq['authority_verbs'])
            if not has_person:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No person/organization detected', 'evidence': []}
            confidence += 0.4
            evidence.append("Person/organization detected")
            if has_title:
                confidence += 0.3
                evidence.append("Authority title present")
            if has_verb:
                confidence += 0.3
                evidence.append("Authority verb present")
            satisfied = confidence >= 0.7

        elif technique == 'Appeal to fear/prejudice':
            fear_count = sum(1 for word in prereq['fear_words'] if word in text_lower)
            has_conditional = any(word in text_lower for word in prereq['conditional_words'])
            if fear_count > 0:
                confidence += min(fear_count * 0.3, 0.6)
                evidence.append(f"{fear_count} fear words")
            if has_conditional:
                confidence += 0.4
                evidence.append("Conditional structure")
            satisfied = confidence >= 0.5

        elif technique == 'Bandwagon':
            crowd_count = sum(1 for word in prereq['crowd_words'] if word in text_lower)
            has_action = any(verb in text_lower for verb in prereq['action_verbs'])
            if crowd_count > 0:
                confidence += min(crowd_count * 0.4, 0.7)
                evidence.append(f"{crowd_count} crowd words")
            if has_action:
                confidence += 0.3
                evidence.append("Action verb present")
            satisfied = confidence >= 0.5

        elif technique == 'Black-and-white Fallacy/Dictatorship':
            binary_found = any(re.search(pattern, text_lower) for pattern in prereq['binary_patterns'])
            has_dichotomy = any(word in text_lower for word in prereq['dichotomy_words'])
            denies_middle = any(phrase in text_lower for phrase in prereq['exclusion_phrases'])
            if binary_found:
                confidence += 0.6
                evidence.append("Binary pattern detected")
            if has_dichotomy:
                confidence += 0.2
                evidence.append("Dichotomy language")
            if denies_middle:
                confidence += 0.2
                evidence.append("Excludes alternatives")
            satisfied = confidence >= 0.5

        elif technique == 'Causal Oversimplification':
            has_causality = any(word in text_lower for word in prereq['causal_words'])
            has_limiting = any(word in text_lower for word in prereq['limiting_words'])
            if not has_causality:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No causal language', 'evidence': []}
            confidence += 0.5
            evidence.append("Causal relationship present")
            if has_limiting:
                confidence += 0.5
                evidence.append("Limiting language")
            satisfied = confidence >= 0.6

        elif technique == 'Doubt':
            doubt_count = sum(1 for word in prereq['doubt_words'] if word in text_lower)
            has_question = any(re.search(pattern, text) for pattern in prereq['questioning_patterns'])
            if doubt_count > 0:
                confidence += min(doubt_count * 0.4, 0.7)
                evidence.append(f"{doubt_count} doubt words")
            if has_question:
                confidence += 0.3
                evidence.append("Questioning language")
            satisfied = confidence >= 0.4

        elif technique == 'Exaggeration/Minimisation':
            exag_count = sum(1 for word in prereq['exaggeration_words'] if word in text_lower)
            super_count = sum(1 for word in prereq['superlatives'] if word in text_lower)
            if exag_count + super_count >= 2:
                confidence += 0.8
                evidence.append(f"{exag_count + super_count} exaggeration indicators")
            elif exag_count + super_count == 1:
                confidence += 0.4
                evidence.append("Some exaggeration")
            satisfied = confidence >= 0.4

        elif technique == 'Flag-waving':
            patriotic_count = sum(1 for word in prereq['patriotic_words'] if word in text_lower)
            if not has_flag and patriotic_count == 0:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No flag or patriotic language', 'evidence': []}
            if has_flag:
                confidence += 0.6
                evidence.append("Flag in image")
            if patriotic_count > 0:
                confidence += min(patriotic_count * 0.3, 0.6)
                evidence.append(f"{patriotic_count} patriotic words")
            satisfied = confidence >= 0.5

        elif technique == 'Glittering generalities (Virtue)':
            virtue_count = sum(1 for word in prereq['virtue_words'] if word in text_lower)
            if virtue_count >= 2:
                confidence += 0.8
                evidence.append(f"{virtue_count} virtue words")
                satisfied = True
            elif virtue_count == 1:
                confidence += 0.4
                evidence.append("One virtue word")

        elif technique == 'Loaded Language':
            loaded_count = sum(1 for word in prereq['loaded_positive'] + prereq['loaded_negative'] if word in text_lower)
            if loaded_count >= prereq['min_loaded_words']:
                confidence = 0.8
                evidence.append(f"{loaded_count} loaded words")
                satisfied = True
            elif loaded_count == 1:
                confidence = 0.4
                evidence.append("Some loaded language")

        elif technique == "Misrepresentation of Someone's Position (Straw Man)":
            has_opponent = any(phrase in text_lower for phrase in prereq['opponent_reference'])
            has_distortion = any(phrase in text_lower for phrase in prereq['distortion_indicators'])
            if not has_opponent:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No opponent reference', 'evidence': []}
            confidence += 0.5
            evidence.append("Opponent referenced")
            if has_distortion:
                confidence += 0.5
                evidence.append("Distortion language")
            satisfied = confidence >= 0.6

        elif technique == 'Name calling/Labeling':
            label_count = sum(1 for word in prereq['negative_labels'] if word in text_lower)
            if label_count > 0:
                confidence = min(label_count * 0.5, 1.0)
                evidence.append(f"{label_count} negative labels")
                satisfied = True

        elif technique == 'Obfuscation, Intentional vagueness, Confusion':
            vague_count = sum(1 for word in prereq['vague_words'] if word in text_lower)
            if vague_count >= 3:
                confidence = 0.7
                evidence.append(f"{vague_count} vague words")
                satisfied = True
            elif vague_count >= 2:
                confidence = 0.4
                evidence.append("Some vague language")

        elif technique == 'Presenting Irrelevant Data (Red Herring)':
            has_shift = any(word in text_lower for word in prereq['topic_shift_indicators'])
            if has_shift:
                confidence = 0.6
                evidence.append("Topic shift detected")
                satisfied = True

        elif technique == 'Reductio ad hitlerum':
            has_nazi = any(word in text_lower for word in prereq['nazi_references'])
            if not has_nazi:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No Nazi/Hitler reference', 'evidence': []}
            confidence = 0.9
            evidence.append("Nazi/Hitler comparison")
            satisfied = True

        elif technique == 'Repetition':
            words = text_lower.split()
            repeated = False
            for i in range(len(words) - 3):
                phrase = ' '.join(words[i:i+3])
                if text_lower.count(phrase) >= 2:
                    repeated = True
                    break
            if repeated:
                confidence = 0.8
                evidence.append("Repeated phrase detected")
                satisfied = True

        elif technique == 'Slogans':
            word_count = len(text.split())
            has_indicators = any(ind in text_lower for ind in prereq['slogan_indicators'])
            if word_count <= prereq['max_words']:
                confidence += 0.5
                evidence.append("Short phrase")
                if has_indicators:
                    confidence += 0.4
                    evidence.append("Slogan indicators")
                satisfied = confidence >= 0.5

        elif technique == 'Smears':
            attack_count = sum(1 for word in prereq['attack_words'] if word in text_lower)
            if attack_count > 0:
                confidence = min(attack_count * 0.5, 1.0)
                evidence.append(f"{attack_count} attack words")
                satisfied = True

        elif technique == 'Thought-terminating cliché':
            has_cliche = any(phrase in text_lower for phrase in prereq['cliche_phrases'])
            if has_cliche:
                confidence = 0.9
                evidence.append("Cliché phrase detected")
                satisfied = True

        elif technique == 'Transfer':
            has_association = any(word in text_lower for word in prereq['association_words'])
            if not has_visual_symbol:
                return {'satisfied': False, 'confidence': 0.0, 'reason': 'No visual symbol for transfer', 'evidence': []}
            confidence += 0.7
            evidence.append("Visual symbol present")
            if has_association:
                confidence += 0.3
                evidence.append("Association language")
            satisfied = True

        elif technique == 'Whataboutism':
            has_whatabout = bool(re.search(prereq['whatabout_pattern'], text_lower))
            has_deflection = any(word in text_lower for word in prereq['deflection_words'])
            if has_whatabout:
                confidence += 0.8
                evidence.append("'What about' pattern detected")
                satisfied = True
            elif has_deflection:
                confidence += 0.3
                evidence.append("Deflection language")

        return {'satisfied': satisfied, 'confidence': confidence, 'reason': f"Prerequisites {'met' if satisfied else 'not met'}", 'evidence': evidence}

    def apply_mutual_exclusions(self, predictions):
        refined = predictions.copy()
        for tech1, prob1 in predictions.items():
            if tech1 in self.mutual_exclusions:
                for tech2 in self.mutual_exclusions[tech1]:
                    if tech2 in predictions:
                        prob2 = predictions[tech2]
                        if prob1 > 0.5 and prob2 > 0.5:
                            if prob1 > prob2:
                                refined[tech2] *= 0.3
                            else:
                                refined[tech1] *= 0.3
        return refined

    def boost_co_occurrences(self, predictions):
        boosted = predictions.copy()
        for tech, prob in predictions.items():
            if tech in self.co_occurrences and prob > 0.6:
                related_techs = self.co_occurrences[tech]['often_with']
                boost_factor = self.co_occurrences[tech]['boost_factor']
                for related in related_techs:
                    if related in boosted:
                        boosted[related] *= boost_factor
                        boosted[related] = min(boosted[related], 0.95)
        return boosted

    def refine_predictions(self, neural_predictions, text, has_person=False, has_visual_symbol=False, has_flag=False, verbose=False):
        refined = neural_predictions.copy()
        for technique in list(refined.keys()):
            prereq_result = self.check_prerequisites(technique, text, has_person, has_visual_symbol, has_flag)
            if not prereq_result['satisfied']:
                old_prob = refined[technique]
                refined[technique] *= 0.1
        refined = self.apply_mutual_exclusions(refined)
        refined = self.boost_co_occurrences(refined)
        return refined

class KnowledgeGraphRefinementLayer:
    def __init__(self, technique_names):
        self.kg = PropagandaKnowledgeGraph()
        self.technique_names = technique_names

    def refine_batch(self, logits, texts, has_persons=None, has_symbols=None, has_flags=None, verbose=False):
        batch_size = logits.shape[0]
        probs = torch.sigmoid(logits)
        if has_persons is None:
            has_persons = [False] * batch_size
        if has_symbols is None:
            has_symbols = [False] * batch_size
        if has_flags is None:
            has_flags = [False] * batch_size
        refined_probs = []
        for i in range(batch_size):
            pred_dict = {self.technique_names[j]: probs[i, j].item() for j in range(len(self.technique_names))}
            refined_dict = self.kg.refine_predictions(pred_dict, texts[i], has_persons[i], has_symbols[i], has_flags[i], verbose=verbose and i == 0)
            refined_prob_vector = torch.tensor([refined_dict[tech] for tech in self.technique_names])
            refined_probs.append(refined_prob_vector)
        refined_probs = torch.stack(refined_probs)
        refined_probs = torch.clamp(refined_probs, 1e-7, 1 - 1e-7)
        refined_logits = torch.log(refined_probs / (1 - refined_probs))
        return refined_logits

class ImprovedMemeClassifier:
    def __init__(self, config, train_df=None):
        self.config = config
        set_seed(config.seed)
        os.makedirs(config.checkpoint_dir, exist_ok=True)
        os.makedirs(config.log_dir, exist_ok=True)
        self.label_to_idx, self.idx_to_label, self.ancestor_matrix, self.num_labels = create_label_mappings()

        self.roberta_model = RobertaModel.from_pretrained(config.roberta_model_name).to(config.device)
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained(config.roberta_model_name)
        for param in self.roberta_model.parameters():
            param.requires_grad = False
        self.roberta_model.eval()

        self.clip_model = CLIPModel.from_pretrained(config.clip_model_name).to(config.device)
        self.clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        self.clip_model.eval()

        self.classifier = ImprovedMultiHeadMLP(input_dim=1792, num_labels=self.num_labels, use_gcn=config.use_gcn, gcn_hidden_dim=config.gcn_hidden_dim, gcn_layers=config.gcn_layers).to(config.device)

        if train_df is not None and config.use_gcn:
            self.adjacency_matrix_graph = build_label_graph(train_df=train_df, label_to_idx=self.label_to_idx, idx_to_label=self.idx_to_label, roberta_model=self.roberta_model, roberta_tokenizer=self.roberta_tokenizer, device=self.config.device, pmi_weight=config.pmi_weight, semantic_weight=config.semantic_weight).to(self.config.device)
        else:
            self.adjacency_matrix_graph = None

        self.optimizer_head1 = Adam([*self.classifier.layer1.parameters(), *self.classifier.layer1_bn.parameters(), *self.classifier.layer2.parameters(), *self.classifier.layer2_bn.parameters(), *self.classifier.head1.parameters(), *self.classifier.head1_reducer.parameters(), *self.classifier.head1_to_head2_residual.parameters()], lr=config.lr)
        self.optimizer_head2 = Adam([*self.classifier.layer3.parameters(), *self.classifier.layer3_bn.parameters(), *self.classifier.layer4.parameters(), *self.classifier.layer4_bn.parameters(), *self.classifier.head2.parameters(), *self.classifier.head2_reducer.parameters(), *self.classifier.head2_to_final_residual.parameters()], lr=config.lr)
        head3_params = [*self.classifier.final_layer1.parameters(), *self.classifier.final_layer1_bn.parameters(), *self.classifier.final_layer2.parameters(), *self.classifier.final_layer2_bn.parameters(), *self.classifier.final_head.parameters(), *self.classifier.final_residual.parameters()]
        if self.classifier.use_gcn:
            head3_params.extend(self.classifier.gcn.parameters())
        self.optimizer_head3 = Adam(head3_params, lr=config.lr)

        self.scheduler_head1 = ReduceLROnPlateau(self.optimizer_head1, patience=config.lr_scheduler_patience, factor=config.lr_scheduler_factor)
        self.scheduler_head2 = ReduceLROnPlateau(self.optimizer_head2, patience=config.lr_scheduler_patience, factor=config.lr_scheduler_factor)
        self.scheduler_head3 = ReduceLROnPlateau(self.optimizer_head3, patience=config.lr_scheduler_patience, factor=config.lr_scheduler_factor)

        self.focal_loss = FocalLoss(alpha=1.0, gamma=2.0)
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.ancestor_matrix = self.ancestor_matrix.to(config.device)
        self.feature_cache = {}
        self.early_stopping = EarlyStopping(patience=config.early_stopping_patience)
        self.optimal_thresholds = None
        self.history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'test_loss': [], 'test_f1': [], 'test_precision': [], 'test_recall': []}

        if config.use_kg_refinement:
            self.technique_names = [self.idx_to_label[i] for i in range(self.num_labels)]
            self.kg_refiner = KnowledgeGraphRefinementLayer(self.technique_names)
            try:
                self.nlp = spacy.load("en_core_web_sm")
            except:
                import subprocess
                subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
                self.nlp = spacy.load("en_core_web_sm")
            print("✓ Knowledge Graph refinement enabled")
        else:
            self.kg_refiner = None
            print("⚠ Knowledge Graph refinement disabled")

    def detect_persons(self, texts):
        has_persons = []
        for text in texts:
            doc = self.nlp(text)
            has_person = any(ent.label_ in ['PERSON', 'ORG'] for ent in doc.ents)
            has_persons.append(has_person)
        return has_persons

    def extract_roberta_features(self, texts):
        with torch.no_grad():
            encoded = self.roberta_tokenizer(texts, padding='max_length', truncation=True, max_length=256, return_tensors='pt')
            input_ids = encoded['input_ids'].to(self.config.device)
            attention_mask = encoded['attention_mask'].to(self.config.device)
            outputs = self.roberta_model(input_ids=input_ids, attention_mask=attention_mask)
            roberta_features = outputs.last_hidden_state[:, 0, :]
            return roberta_features

    def extract_clip_features(self, texts, images):
        with torch.no_grad():
            inputs = self.clip_processor(text=texts, images=images, return_tensors='pt', padding='max_length', truncation=True, max_length=77)
            for key in inputs:
                inputs[key] = inputs[key].to(self.config.device)
            outputs = self.clip_model(**inputs)
            clip_features = torch.cat((outputs.text_embeds, outputs.image_embeds), dim=-1)
            return clip_features

    def extract_features(self, texts, images):
        roberta_features = self.extract_roberta_features(list(texts))
        clip_features = self.extract_clip_features(list(texts), list(images))
        combined_features = torch.cat((roberta_features, clip_features), dim=-1)
        return combined_features

    def precompute_features(self, dataset, cache_name, batch_size=32):
        features_list = []
        labels_list = []
        num_samples = len(dataset)
        num_batches = (num_samples + batch_size - 1) // batch_size
        with torch.no_grad():
            for batch_idx in tqdm(range(num_batches), desc=f"Extracting {cache_name} features"):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, num_samples)
                batch_texts = []
                batch_images = []
                batch_labels = []
                for idx in range(start_idx, end_idx):
                    text, image, label, _ = dataset[idx]
                    batch_texts.append(text)
                    batch_images.append(image)
                    batch_labels.append(label)
                features = self.extract_features(batch_texts, batch_images)
                labels_tensor = torch.stack(batch_labels)
                features_list.append(features.cpu())
                labels_list.append(labels_tensor.cpu())
        all_features = torch.cat(features_list, dim=0)
        all_labels = torch.cat(labels_list, dim=0)
        self.feature_cache[cache_name] = {'features': all_features, 'labels': all_labels}
        return all_features, all_labels

    def collate_fn_cached(self, batch):
        indices = [item[3] for item in batch]
        cache_name = getattr(self, '_current_cache', None)
        if cache_name and cache_name in self.feature_cache:
            cached_data = self.feature_cache[cache_name]
            features = cached_data['features'][indices]
            labels = cached_data['labels'][indices]
        else:
            texts, images, labels_list, _ = zip(*batch)
            labels = torch.stack(labels_list)
            features = self.extract_features(list(texts), list(images))
        return features.to(self.config.device), labels.to(self.config.device), indices

    def create_hierarchical_targets(self, labels):
        batch_size = labels.shape[0]
        ethos_idx = self.label_to_idx['Ethos']
        pathos_idx = self.label_to_idx['Pathos']
        logos_idx = self.label_to_idx['Logos']
        head1_targets = torch.zeros(batch_size, 3)
        head1_targets[:, 0] = labels[:, ethos_idx]
        head1_targets[:, 1] = labels[:, pathos_idx]
        head1_targets[:, 2] = labels[:, logos_idx]
        ad_hominem_idx = self.label_to_idx['Ad Hominem']
        justification_idx = self.label_to_idx['Justification']
        distraction_idx = self.label_to_idx['Distraction']
        simplification_idx = self.label_to_idx['Simplification']
        other_idx = self.label_to_idx['Other']
        head2_targets = torch.zeros(batch_size, 5)
        head2_targets[:, 0] = labels[:, ad_hominem_idx]
        head2_targets[:, 1] = labels[:, justification_idx]
        head2_targets[:, 2] = labels[:, distraction_idx]
        head2_targets[:, 3] = labels[:, simplification_idx]
        head2_targets[:, 4] = labels[:, other_idx]
        return head1_targets, head2_targets

    def create_progressive_class_masks(self, train_dataset):
        class_counts = torch.zeros(self.num_labels)
        for i in range(len(train_dataset)):
            _, _, labels, _ = train_dataset[i]
            class_counts += labels
        high_freq_threshold = 100
        med_freq_threshold = 20
        high_freq_mask = class_counts >= high_freq_threshold
        med_freq_mask = class_counts >= med_freq_threshold
        all_classes_mask = torch.ones(self.num_labels, dtype=torch.bool)
        return high_freq_mask, med_freq_mask, all_classes_mask

    def train_phase_progressive(self, train_loader, phase, class_mask, epochs_for_phase=3):
        if phase == 'head1':
            self.classifier.freeze_head2_layers()
            self.classifier.freeze_head3_layers()
            self.classifier.unfreeze_head1_layers()
            optimizer = self.optimizer_head1
            scheduler = self.scheduler_head1
        elif phase == 'head2':
            self.classifier.freeze_head1_layers()
            self.classifier.freeze_head3_layers()
            self.classifier.unfreeze_head2_layers()
            optimizer = self.optimizer_head2
            scheduler = self.scheduler_head2
        elif phase == 'head3':
            self.classifier.freeze_head1_layers()
            self.classifier.freeze_head2_layers()
            self.classifier.unfreeze_head3_layers()
            optimizer = self.optimizer_head3
            scheduler = self.scheduler_head3
        else:
            raise ValueError(f"Unknown phase: {phase}")
        self.classifier.train()
        for epoch in range(epochs_for_phase):
            total_loss = 0
            num_batches = 0
            pbar = tqdm(train_loader, desc=f"Phase {phase.upper()} - Epoch {epoch+1}/{epochs_for_phase}")
            for features, labels, _ in pbar:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)
                masked_labels = labels * class_mask.to(self.config.device)
                optimizer.zero_grad()
                if phase == 'head3' and self.adjacency_matrix_graph is not None:
                    output1, output2, output_final = self.classifier(features, adjacency_matrix=self.adjacency_matrix_graph, training_phase=phase)
                else:
                    output1, output2, output_final = self.classifier(features, training_phase=phase)
                if phase == 'head1':
                    head1_targets, _ = self.create_hierarchical_targets(masked_labels)
                    head1_targets = head1_targets.to(self.config.device)
                    focal_loss = self.focal_loss(output1, head1_targets)
                    loss = focal_loss
                elif phase == 'head2':
                    _, head2_targets = self.create_hierarchical_targets(masked_labels)
                    head2_targets = head2_targets.to(self.config.device)
                    focal_loss = self.focal_loss(output2, head2_targets)
                    loss = focal_loss
                elif phase == 'head3':
                    focal_loss = self.focal_loss(output_final, masked_labels)
                    hierarchy_loss = hierarchical_consistency_loss(output_final, self.ancestor_matrix, lambda_consistency=0.1)
                    loss = focal_loss + hierarchy_loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.classifier.parameters(), self.config.gradient_clip_norm)
                optimizer.step()
                total_loss += loss.item()
                num_batches += 1
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            avg_loss = total_loss / num_batches
            scheduler.step(avg_loss)

    def validate_with_kg_refinement(self, val_loader, dataset_name="Validation", update_thresholds=False):
        self.classifier.eval()
        total_loss = 0
        all_logits_neural = []
        all_logits_refined = []
        all_labels = []
        all_texts = []
        all_images = []

        with torch.no_grad():
            for batch_idx, (features, labels, indices) in enumerate(tqdm(val_loader, desc=f"Collecting predictions")):
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)
                batch_texts = []
                batch_images = []
                for idx in indices:
                    dataset = val_loader.dataset
                    text, image, _, _ = dataset[idx]
                    batch_texts.append(text)
                    batch_images.append(image)
                if self.adjacency_matrix_graph is not None:
                    output1, output2, output_final = self.classifier(features, adjacency_matrix=self.adjacency_matrix_graph, training_phase='all')
                else:
                    output1, output2, output_final = self.classifier(features, training_phase='all')
                focal_loss = self.focal_loss(output_final, labels)
                hierarchy_loss = hierarchical_consistency_loss(output_final, self.ancestor_matrix, lambda_consistency=0.1)
                loss = focal_loss + hierarchy_loss
                total_loss += loss.item()
                all_logits_neural.append(output_final.cpu())
                all_labels.append(labels.cpu())
                all_texts.extend(batch_texts)
                all_images.extend(batch_images)

        all_logits_neural = torch.cat(all_logits_neural)
        all_labels = torch.cat(all_labels)

        if self.kg_refiner is not None:
            print("\n" + "="*70)
            print("APPLYING KNOWLEDGE GRAPH REFINEMENT")
            print("="*70)
            has_persons = self.detect_persons(all_texts)
            has_symbols = [False] * len(all_texts)
            has_flags = [False] * len(all_texts)
            batch_size = 32
            num_samples = len(all_logits_neural)
            refined_logits_list = []
            for start_idx in tqdm(range(0, num_samples, batch_size), desc="Refining"):
                end_idx = min(start_idx + batch_size, num_samples)
                batch_logits = all_logits_neural[start_idx:end_idx]
                batch_texts = all_texts[start_idx:end_idx]
                batch_persons = has_persons[start_idx:end_idx]
                batch_symbols = has_symbols[start_idx:end_idx]
                batch_flags = has_flags[start_idx:end_idx]
                refined_batch = self.kg_refiner.refine_batch(batch_logits, batch_texts, batch_persons, batch_symbols, batch_flags, verbose=False)
                refined_logits_list.append(refined_batch)
            all_logits_refined = torch.cat(refined_logits_list)
            neural_probs = torch.sigmoid(all_logits_neural)
            refined_probs = torch.sigmoid(all_logits_refined)
            avg_change = (refined_probs - neural_probs).abs().mean().item()
            print(f"\n✓ KG Refinement complete! Average change: {avg_change:.4f}")
        else:
            all_logits_refined = all_logits_neural

        all_probs = torch.sigmoid(all_logits_refined)
        if update_thresholds:
            self.optimal_thresholds = find_optimal_thresholds(all_labels.numpy(), all_probs.numpy(), self.num_labels)
        if self.optimal_thresholds is not None:
            y_pred = torch.zeros_like(all_probs)
            for i, threshold in enumerate(self.optimal_thresholds):
                y_pred[:, i] = (all_probs[:, i] > threshold).float()
        else:
            y_pred = (all_probs > 0.5).float()
        f1, precision, recall = hierarchical_f1_score(all_logits_refined, all_labels, self.ancestor_matrix.cpu())
        avg_loss = total_loss / len(val_loader)
        y_pred_np = y_pred.int().numpy()
        y_true_np = all_labels.int().numpy()
        target_names = [self.idx_to_label[i] for i in range(self.num_labels)]
        print(f"\n--- {dataset_name} Classification Report ---")
        report = classification_report(y_true_np, y_pred_np, target_names=target_names, zero_division=0)
        print(report)
        return avg_loss, f1, precision, recall

    def fit_improved(self, train_loader, val_loader, test_loader=None, use_cached_features=True):
        if use_cached_features:
            self._current_cache = 'train'
        train_dataset = train_loader.dataset
        high_freq_mask, med_freq_mask, all_classes_mask = self.create_progressive_class_masks(train_dataset)

        self.train_phase_progressive(train_loader, 'head1', high_freq_mask, epochs_for_phase=3)
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_kg_refinement(val_loader, "Validation after Head 1 (High Freq)", update_thresholds=True)

        if use_cached_features:
            self._current_cache = 'train'
        self.train_phase_progressive(train_loader, 'head1', med_freq_mask, epochs_for_phase=2)
        self.train_phase_progressive(train_loader, 'head1', all_classes_mask, epochs_for_phase=2)
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_kg_refinement(val_loader, "Validation after Complete Head 1", update_thresholds=True)

        if use_cached_features:
            self._current_cache = 'train'
        self.train_phase_progressive(train_loader, 'head2', high_freq_mask, epochs_for_phase=2)
        self.train_phase_progressive(train_loader, 'head2', med_freq_mask, epochs_for_phase=2)
        self.train_phase_progressive(train_loader, 'head2', all_classes_mask, epochs_for_phase=2)
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_kg_refinement(val_loader, "Validation after Head 2", update_thresholds=True)

        if use_cached_features:
            self._current_cache = 'train'
        self.train_phase_progressive(train_loader, 'head3', high_freq_mask, epochs_for_phase=3)
        self.train_phase_progressive(train_loader, 'head3', med_freq_mask, epochs_for_phase=3)
        self.train_phase_progressive(train_loader, 'head3', all_classes_mask, epochs_for_phase=4)

        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_kg_refinement(val_loader, "Final Validation", update_thresholds=True)

        if test_loader is not None:
            if use_cached_features:
                self._current_cache = 'test'
            test_loss, test_f1, test_precision, test_recall = self.validate_with_kg_refinement(test_loader, "Final Test", update_thresholds=False)
            print(f"\n{'='*60}")
            print(f"FINAL RESULTS")
            print(f"{'='*60}")
            print(f"Validation - F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}")
            print(f"Test - F1: {test_f1:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}")
            print(f"{'='*60}")

        self.save_checkpoint(f'complete_model_f1_{val_f1:.4f}.pth')

    def save_checkpoint(self, filename):
        checkpoint = {
            'classifier_state_dict': self.classifier.state_dict(),
            'optimizer_head1_state_dict': self.optimizer_head1.state_dict(),
            'optimizer_head2_state_dict': self.optimizer_head2.state_dict(),
            'optimizer_head3_state_dict': self.optimizer_head3.state_dict(),
            'label_to_idx': self.label_to_idx,
            'idx_to_label': self.idx_to_label,
            'ancestor_matrix': self.ancestor_matrix,
            'num_labels': self.num_labels,
            'history': self.history,
            'optimal_thresholds': self.optimal_thresholds,
            'adjacency_matrix_graph': self.adjacency_matrix_graph
        }
        torch.save(checkpoint, os.path.join(self.config.checkpoint_dir, filename))

def filter_existing_images(data, img_dir):
    """Filter dataset to only include entries with existing images"""
    img_dir_path = Path(img_dir)
    filtered = []
    missing = 0
    for entry in data:
        if 'image' not in entry:
            filtered.append(entry)
            continue
        img_path = img_dir_path / entry['image']
        if img_path.exists():
            filtered.append(entry)
        else:
            missing += 1
    if missing > 0:
        print(f"  ⚠ Filtered out {missing} samples with missing images")
    return filtered

def main():
    config = CFG()

    print("="*70)
    print("LOADING DATASETS")
    print("="*70)

    with open(config.train_json) as fp:
        train = json.load(fp)
    print(f"Train JSON loaded: {len(train)} samples")
    train = filter_existing_images(train, config.train_img_dir)
    print(f"Train after filtering: {len(train)} samples")

    with open(config.val_json) as fp:
        valid = json.load(fp)
    print(f"Validation JSON loaded: {len(valid)} samples")
    valid = filter_existing_images(valid, config.val_img_dir)
    print(f"Validation after filtering: {len(valid)} samples")

    test = None
    if os.path.exists(config.test_json):
        with open(config.test_json) as fp:
            test = json.load(fp)
        print(f"Test JSON loaded: {len(test)} samples")
        test = filter_existing_images(test, config.test_img_dir)
        print(f"Test after filtering: {len(test)} samples")

    train_df = pd.DataFrame(train)
    valid_df = pd.DataFrame(valid)
    test_df = pd.DataFrame(test) if test is not None else None

    classifier = ImprovedMemeClassifier(config, train_df=train_df)

    train_dataset = MemeDataset(train_df, config.train_img_dir, classifier.clip_processor, classifier.label_to_idx, classifier.ancestor_matrix, use_caption=config.use_caption, caption_separator=config.caption_separator)
    val_dataset = MemeDataset(valid_df, config.val_img_dir, classifier.clip_processor, classifier.label_to_idx, classifier.ancestor_matrix, use_caption=config.use_caption, caption_separator=config.caption_separator)
    test_dataset = None
    if test_df is not None:
        test_dataset = MemeDataset(test_df, config.test_img_dir, classifier.clip_processor, classifier.label_to_idx, classifier.ancestor_matrix, use_caption=config.use_caption, caption_separator=config.caption_separator)

    classifier.precompute_features(train_dataset, 'train', batch_size=config.batch_size)
    classifier.precompute_features(val_dataset, 'val', batch_size=config.batch_size)
    if test_dataset is not None:
        classifier.precompute_features(test_dataset, 'test', batch_size=config.batch_size)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=classifier.collate_fn_cached, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=classifier.collate_fn_cached, num_workers=0)
    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=classifier.collate_fn_cached, num_workers=0)

    classifier.fit_improved(train_loader, val_loader, test_loader, use_cached_features=True)

if __name__ == "__main__":
    main()

LOADING DATASETS
Train JSON loaded: 7000 samples
Train after filtering: 7000 samples
Validation JSON loaded: 500 samples
Validation after filtering: 500 samples
Test JSON loaded: 1000 samples
Test after filtering: 1000 samples


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Extracting embeddings: 100%|██████████| 31/31 [00:00<00:00, 51.06it/s]


✓ Knowledge Graph refinement enabled


Phase HEAD1 - Epoch 3/3: 100%|██████████| 219/219 [01:54<00:00,  1.91it/s, loss=0.1080]



APPLYING KNOWLEDGE GRAPH REFINEMENT


...

Refining: 100%|██████████| 16/16 [11:01<00:00, 41.33s/it]



✓ KG Refinement complete! Average change: 0.1267

--- Final Validation Classification Report ---
                                                     precision    recall  f1-score   support

                                         Ad Hominem       0.75      0.92      0.83       340
                        Appeal to (Strong) Emotions       0.20      0.41      0.27        27
                                Appeal to authority       0.34      0.26      0.29        66
                           Appeal to fear/prejudice       0.19      0.47      0.27        34
                                          Bandwagon       0.04      0.12      0.06         8
               Black-and-white Fallacy/Dictatorship       0.29      0.51      0.37        55
                          Causal Oversimplification       0.20      0.18      0.19        22
                                        Distraction       0.17      0.38      0.23        34
                                              Doubt       0.13  


APPLYING KNOWLEDGE GRAPH REFINEMENT


Refining: 100%|██████████| 32/32 [09:40<00:00, 18.15s/it]


✓ KG Refinement complete! Average change: 0.1385

--- Final Test Classification Report ---
                                                     precision    recall  f1-score   support

                                         Ad Hominem       0.72      0.97      0.82       687
                        Appeal to (Strong) Emotions       0.21      0.55      0.30        56
                                Appeal to authority       0.58      0.43      0.49       143
                           Appeal to fear/prejudice       0.26      0.42      0.32        78
                                          Bandwagon       0.05      0.28      0.08        18
               Black-and-white Fallacy/Dictatorship       0.26      0.44      0.33       103
                          Causal Oversimplification       0.44      0.07      0.12        56
                                        Distraction       0.25      0.43      0.32        83
                                              Doubt       0.14      0.